# Inferência — identidade ao longo do tempo (entregável do enunciado)

Recebe o caminho de uma sequência no formato MOT17 (`seqinfo.ini` + `img1/`, com
`det/det_<VARIANTE>.txt` e/ou `gt/gt.txt` opcionais) e devolve:

1. um vídeo `.mp4` com as caixas coloridas por identidade, **cor consistente da mesma
   track do início ao fim** (não é cor por classe, é cor por ID);
2. a contagem de objetos únicos rastreados na sequência.

**Roda sem retreinar**: carrega o checkpoint `checkpoints/motion_gru_mot17.pt` (modelo
final da Parte 2/Parte 3, GRU) e só faz inferência. Troque `SEQ_DIR` na célula de
parâmetros abaixo para apontar para qualquer outra sequência MOT17-like já baixada em
`data/MOT17/<split>/<nome>` — nenhuma outra célula precisa mudar.

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False and Path.cwd().name == "notebooks" else Path.cwd()
# garante ROOT = raiz do repo mesmo se o notebook for aberto de dentro de notebooks/
if (ROOT / "pa2").exists() is False:
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import colorsys
import time

import imageio.v2 as imageio
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont

from pa2.models.motion_rnn import MotionGRU
from pa2.motion_tracker import MotionRNNTracker
from pa2.mot17 import MOT17Sequence

print(f"ROOT = {ROOT}")

ROOT = C:\Users\Matheus Pinto\Documents\FGV\graduacao\6o_periodo\DL\PA2


## Parâmetros

`SEQ_DIR` é a única coisa que precisa mudar para rodar em outra sequência. Se a pasta
tiver um `det/det_<DET_VARIANT>.txt` (padrão MOT17, como as 4 sequências baixadas pelo
projeto), usamos essa detecção pública — é o caminho rápido (segundos para a sequência
inteira, já que o tracker em si é só um GRU pequeno). Se não houver `det.txt` (uma
sequência de fato nova, sem anotação nenhuma), caímos automaticamente no detector
pré-treinado do torchvision (`pa2/torch_detector.py`, Faster R-CNN) — mais genérico
(funciona em qualquer vídeo de pessoas), mas ~5-6s por quadro em CPU, então nesse
caso limitamos a `MAX_FRAMES_FALLBACK` quadros por padrão.

In [2]:
SEQ_DIR = ROOT / "data" / "MOT17" / "val" / "MOT17-09"  # <- troque aqui para qualquer outra sequência
DET_VARIANT = "SDP"  # variante do detector público (DPM, FRCNN ou SDP) usada se existir det.txt
CKPT_PATH = ROOT / "checkpoints" / "motion_gru_mot17.pt"
HIDDEN_SIZE = 64
MAX_AGE = 15  # mesmo valor usado nas Partes 4 e 5, calibrado para o frame rate do MOT17 (~30fps)

MAX_FRAMES = None  # None = sequência inteira; int = só os N primeiros quadros (preview rápido)
MAX_FRAMES_FALLBACK = 150  # teto quando cai no detector torchvision (sem det.txt disponível)

OUT_DIR = ROOT / "outputs" / "inferencia"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Carrega a sequência e as detecções

In [3]:
seq = MOT17Sequence(SEQ_DIR)
# nota: prints ficam sem acento de proposito -- o kernel Jupyter neste ambiente
# Windows corrompe caracteres acentuados no stdout capturado pelo nbconvert
# (bug de console/codepage, nao do nosso codigo); comentarios e markdown
# acentuados ficam intactos pois sao so texto-fonte, nunca passam por stdout.
print(f"Sequencia: {seq.name}  ({seq.n_frames} quadros, {seq.info['imWidth']}x{seq.info['imHeight']}, "
      f"{seq.info['frameRate']}fps)")

det_path = SEQ_DIR / "det" / f"det_{DET_VARIANT}.txt"
if not det_path.exists():
    det_path = SEQ_DIR / "det" / "det.txt"
use_fallback_detector = not det_path.exists()

n_frames_to_run = MAX_FRAMES if MAX_FRAMES is not None else seq.n_frames
if use_fallback_detector:
    n_frames_to_run = min(n_frames_to_run, MAX_FRAMES_FALLBACK)
    print(f"[aviso] sem det.txt em {SEQ_DIR} -- usando o detector torchvision (Faster R-CNN) "
          f"nos primeiros {n_frames_to_run} quadros.")
else:
    print(f"Deteccao publica encontrada: {det_path.name}")

W, H = seq.info["imWidth"], seq.info["imHeight"]
det_by_frame, score_by_frame = {}, {}

if use_fallback_detector:
    from pa2.torch_detector import detect_persons, load_person_detector

    model_det, preprocess = load_person_detector()
    t0 = time.time()
    for f in range(1, n_frames_to_run + 1):
        img = np.array(Image.open(seq.frame_path(f)).convert("RGB"))
        dets = detect_persons(model_det, preprocess, img, score_threshold=0.5)
        boxes = dets[:, :4] if len(dets) else np.zeros((0, 4))
        scores = dets[:, 4] if len(dets) else np.zeros(0)
        det_by_frame[f] = np.array([[x / W, y / H, w / W, h / H] for x, y, w, h in boxes], dtype=np.float32)
        score_by_frame[f] = scores.astype(np.float32)
    print(f"  detector torchvision: {time.time() - t0:.1f}s para {n_frames_to_run} quadros")
else:
    from pa2.mot17 import load_det

    raw = load_det(det_path)
    by_frame = {}
    for f, x, y, w, h, s in raw:
        by_frame.setdefault(int(f), []).append((x, y, w, h, s))
    for f in range(1, n_frames_to_run + 1):
        rows = by_frame.get(f, [])
        det_by_frame[f] = np.array([[x / W, y / H, w / W, h / H] for x, y, w, h, s in rows], dtype=np.float32)
        score_by_frame[f] = np.array([s for *_, s in rows], dtype=np.float32)

Sequencia: MOT17-09-FRCNN  (525 quadros, 1920x1080, 30fps)
Deteccao publica encontrada: det_SDP.txt


## Carrega o modelo (checkpoint, sem retreinar) e roda o tracker

In [4]:
model = MotionGRU(hidden_size=HIDDEN_SIZE)
model.load_state_dict(torch.load(CKPT_PATH, weights_only=True, map_location="cpu"))
model.eval()

tracker = MotionRNNTracker(model, iou_threshold=0.3, max_age=MAX_AGE, match_method="hungarian")
pred_norm = tracker.run(det_by_frame, score_by_frame)
pred_tracks = [(f, i, x * W, y * H, w * W, h * H) for f, i, x, y, w, h in pred_norm]

tracks_by_frame = {}
for f, i, x, y, w, h in pred_tracks:
    tracks_by_frame.setdefault(f, []).append((i, x, y, w, h))

unique_ids = sorted({i for _, i, *_ in pred_tracks})
print(f"Objetos unicos rastreados: {len(unique_ids)}")
print(f"Total de caixas preditas: {len(pred_tracks)}  em {n_frames_to_run} quadros")

Objetos unicos rastreados: 70
Total de caixas preditas: 3607  em 525 quadros


## Cor consistente por identidade

Cor determinística a partir do ID (roda-cor via razão áurea) — a mesma track tem a
mesma cor do primeiro ao último quadro em que aparece, e IDs diferentes ficam bem
distribuídos nas cores mesmo com dezenas de identidades (não se repete a cada 10 como
um colormap categórico cíclico faria).

In [5]:
def id_color(track_id: int) -> tuple[int, int, int]:
    h = (track_id * 0.6180339887498949) % 1.0
    r, g, b = colorsys.hsv_to_rgb(h, 0.80, 0.95)
    return int(r * 255), int(g * 255), int(b * 255)


try:
    font = ImageFont.truetype("arial.ttf", 20)
except OSError:
    font = ImageFont.load_default()

## Desenha os quadros e salva o vídeo

In [6]:
out_path = OUT_DIR / f"{seq.name}_tracked.mp4"
writer = imageio.get_writer(str(out_path), fps=seq.info["frameRate"], codec="libx264", quality=7)

seen_so_far = set()
for f in range(1, n_frames_to_run + 1):
    img = Image.open(seq.frame_path(f)).convert("RGB")
    draw = ImageDraw.Draw(img)
    for tid, x, y, w, h in tracks_by_frame.get(f, []):
        color = id_color(tid)
        seen_so_far.add(tid)
        draw.rectangle([x, y, x + w, y + h], outline=color, width=3)
        label = f"ID {tid}"
        draw.rectangle([x, y - 22, x + 10 + 11 * len(label), y], fill=color)
        draw.text((x + 4, y - 21), label, fill="black", font=font)

    draw.text((10, 10), f"quadro {f}/{n_frames_to_run}  |  objetos únicos até aqui: {len(seen_so_far)}",
               fill="white", font=font, stroke_width=2, stroke_fill="black")
    writer.append_data(np.array(img))

writer.close()
print(f"[ok] video salvo em {out_path}")
print(f"Contagem final de objetos unicos: {len(unique_ids)}")

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1920, 1080) to (1920, 1088) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


[ok] video salvo em C:\Users\Matheus Pinto\Documents\FGV\graduacao\6o_periodo\DL\PA2\outputs\inferencia\MOT17-09-FRCNN_tracked.mp4
Contagem final de objetos unicos: 70


## Reprodução no notebook

In [7]:
from IPython.display import Video

# caminho relativo ao diretorio de trabalho do kernel, para que o link
# funcione em qualquer maquina que clonar o repositorio, nao so na nossa
rel_video_path = Path(os.path.relpath(out_path, Path.cwd())).as_posix()
Video(rel_video_path, embed=False, width=800)